### Molecular Structure & Property Explorer — An RDKit-Based Cheminformatics Application

### 01. Environment Setup

* Python environment
* RDKit
* NumPy
* Pandas
* Matplotlib
* Jupyter Notebook

In [3]:
%%capture
%pip install pandas numpy matplotlib seaborn rdkit

In [4]:
# Core data science
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

print("Environment ready!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("RDKit:", Chem.rdBase.rdkitVersion)

Environment ready!
Pandas: 3.0.5
NumPy: 2.5.3
RDKit: 2026.03.6


### 02. Load ESOL Dataset

* Load CSV
* Inspect dataset
* Identify target/property columns

In [5]:
import pandas as pd

data = pd.read_csv("data\delaney-processed.csv")

data.head()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\rasel\AppData\Local\Temp\ipykernel_8324\942561932.py:3: SyntaxWarning: invalid escape sequence '\d'
  data = pd.read_csv("data\delaney-processed.csv")


,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


| Column                          | What it represents                 | Category              |
| ------------------------------- | ---------------------------------- | --------------------- |
| `Compound ID`                   | Molecule name/identifier           | Identification        |
| `ESOL predicted log solubility` | Model-predicted logS               | Prediction            |
| `Minimum Degree`                | Minimum atom connectivity degree   | Graph descriptor      |
| `Molecular Weight`              | Molecular mass                     | Property              |
| `Number of H-Bond Donors`       | H-bond donating groups             | Property              |
| `Number of Rings`               | Ring count                         | Structural descriptor |
| `Number of Rotatable Bonds`     | Molecular flexibility              | Structural descriptor |
| `Polar Surface Area`            | Molecular polarity                 | Property              |
| `measured log solubility`       | Experimental/reference logS        | Target                |
| `smiles`                        | Molecular structure representation | Representation        |


### Dataset Inspection

In [6]:
# Column names
data.columns

Index(['Compound ID', 'ESOL predicted log solubility in mols per litre',
       'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors',
       'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area',
       'measured log solubility in mols per litre', 'smiles'],
      dtype='str')

In [7]:
# Column names as Python list
data.columns.tolist()

['Compound ID',
 'ESOL predicted log solubility in mols per litre',
 'Minimum Degree',
 'Molecular Weight',
 'Number of H-Bond Donors',
 'Number of Rings',
 'Number of Rotatable Bonds',
 'Polar Surface Area',
 'measured log solubility in mols per litre',
 'smiles']

In [8]:
print(len(data.columns))

10


In [9]:
# Dataset information
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 10 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Compound ID                                      1128 non-null   str    
 1   ESOL predicted log solubility in mols per litre  1128 non-null   float64
 2   Minimum Degree                                   1128 non-null   int64  
 3   Molecular Weight                                 1128 non-null   float64
 4   Number of H-Bond Donors                          1128 non-null   int64  
 5   Number of Rings                                  1128 non-null   int64  
 6   Number of Rotatable Bonds                        1128 non-null   int64  
 7   Polar Surface Area                               1128 non-null   float64
 8   measured log solubility in mols per litre        1128 non-null   float64
 9   smiles                                   

In [10]:
# Data types
data.dtypes

Compound ID                                            str
ESOL predicted log solubility in mols per litre    float64
Minimum Degree                                       int64
Molecular Weight                                   float64
Number of H-Bond Donors                              int64
Number of Rings                                      int64
Number of Rotatable Bonds                            int64
Polar Surface Area                                 float64
measured log solubility in mols per litre          float64
smiles                                                 str
dtype: object

In [11]:
# Missing values
data.isnull().sum()

Compound ID                                        0
ESOL predicted log solubility in mols per litre    0
Minimum Degree                                     0
Molecular Weight                                   0
Number of H-Bond Donors                            0
Number of Rings                                    0
Number of Rotatable Bonds                          0
Polar Surface Area                                 0
measured log solubility in mols per litre          0
smiles                                             0
dtype: int64

In [12]:
# Dataset dimensions
data.shape

(1128, 10)

In [13]:
# Statistical summary
data.describe()

,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre
count,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000,1128.000000
mean,-2.988192,1.058511,203.937074,0.701241,1.390957,2.177305,34.872881,-3.050102
std,1.683220,0.238560,102.738077,1.089727,1.318286,2.640974,35.383593,2.096441
min,-9.702000,0.000000,16.043000,0.000000,0.000000,0.000000,0.000000,-11.600000
25%,-3.948250,1.000000,121.183000,0.000000,0.000000,0.000000,0.000000,-4.317500
50%,-2.870000,1.000000,182.179000,0.000000,1.000000,1.000000,26.300000,-2.860000
75%,-1.843750,1.000000,270.372000,1.000000,2.000000,3.000000,55.440000,-1.600000
max,1.091000,2.000000,780.949000,11.000000,8.000000,23.000000,268.680000,1.580000


In [14]:
# Duplicate rows
data.duplicated().sum()

np.int64(0)

In [15]:
# Duplicate molecules based on SMILES
data["smiles"].duplicated().sum()

np.int64(0)

### SMILES

> SMILES = Simplified Molecular Input Line Entry System
>
> This is the text representation of the molecular structure.
>
> From SMILES, you can generate:

* 2D structure
* molecular properties
* descriptors
* fingerprints
* molecular similarity
* molecular graph
* drug-likeness calculations

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Draw,
    Descriptors,
    Crippen,
    Lipinski,
    rdMolDescriptors,
    QED
)

from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys

In [19]:
# data["is_valid"] = data["mol"].notna()

# valid_data = data[data["is_valid"]].copy()

print("Total records:", len(data))
print("Valid molecules:", len(data))
# print("Invalid molecules:", (~data["is_valid"]).sum())

Total records: 1128
Valid molecules: 1128


In [20]:
# 2D Molecular Structure
from rdkit.Chem import rdDepictor

def generate_2d(mol):
    mol = Chem.Mol(mol)
    rdDepictor.Compute2DCoords(mol)
    return mol

In [21]:
data["mol_2d"] = data["mol"].apply(generate_2d)

KeyError: 'mol'

In [ ]:
# Display one molecule
mol = data.iloc[0]["mol_2d"]

Draw.MolToImage(
    mol,
    size=(500, 500)
)

In [ ]:
# Display multiple molecules

mols = data["mol_2d"].head(12).tolist()

legends = data["Compound ID"].head(12).tolist()

Draw.MolsToGridImage(
    mols,
    legends=legends,
    molsPerRow=4,
    subImgSize=(300, 300)
)

In [ ]:
# Molecular Properties
def calculate_molecular_properties(mol):
    return {
        "MolecularWeight": Descriptors.MolWt(mol),
        "ExactMolWeight": Descriptors.ExactMolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "HeavyAtomCount": Lipinski.HeavyAtomCount(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRingCount": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "FormalCharge": Chem.GetFormalCharge(mol),
        "MolMR": Crippen.MolMR(mol),
    }


In [ ]:
properties = data["mol"].apply(
    calculate_molecular_properties
)

properties_df = pd.DataFrame(
    properties.tolist(),
    index=data.index
)

data = pd.concat(
    [data, properties_df],
    axis=1
)

In [ ]:
data[
    [
        "Compound ID",
        "MolecularWeight",
        "LogP",
        "TPSA",
        "HBD",
        "HBA",
        "RotatableBonds",
        "HeavyAtomCount",
        "RingCount",
        "AromaticRingCount",
        "FractionCSP3",
        "FormalCharge",
        "MolMR"
    ]
].head()

In [ ]:
# RDKit Descriptor Set

def calculate_all_descriptors(mol):
    return Descriptors.CalcMolDescriptors(
        mol,
        missingVal=np.nan
    )

In [ ]:
descriptor_dicts = data["mol"].apply(
    calculate_all_descriptors
)

In [ ]:
descriptor_df = pd.DataFrame(
    descriptor_dicts.tolist(),
    index=data.index
)

In [ ]:
descriptor_df.shape

In [ ]:
# Inspect descriptor names

descriptor_df.columns.tolist()

In [ ]:
descriptor_df.head()

### Descriptors

- EState
- Chi
- Kappa
- BalabanJ
- BertzCT
- MolMR
- Fraction Csp3

In [ ]:
advanced_descriptor_names = [
    "BalabanJ",
    "BertzCT",
    "MolMR",
    "FractionCSP3",
    "Chi0n",
    "Chi1n",
    "Chi2n",
    "Chi3n",
    "Chi4n",
    "Kappa1",
    "Kappa2",
    "Kappa3"
]

In [ ]:
available = [
    col
    for col in advanced_descriptor_names
    if col in descriptor_df.columns
]

descriptor_df[available].head()

### Extract atoms

In [ ]:
def atom_features(mol):
    features = []

    for atom in mol.GetAtoms():
        features.append({
            "atom_index": atom.GetIdx(),
            "symbol": atom.GetSymbol(),
            "atomic_number": atom.GetAtomicNum(),
            "degree": atom.GetDegree(),
            "formal_charge": atom.GetFormalCharge(),
            "total_h": atom.GetTotalNumHs(),
            "hybridization": str(atom.GetHybridization()),
            "aromatic": atom.GetIsAromatic(),
            "in_ring": atom.IsInRing(),
        })

    return pd.DataFrame(features)

In [ ]:
mol = valid_data.iloc[0]["mol"]

atom_df = atom_features(mol)

atom_df

### Molecular Graph Edges

In [ ]:
def bond_features(mol):
    bonds = []

    for bond in mol.GetBonds():
        bonds.append({
            "begin_atom": bond.GetBeginAtomIdx(),
            "end_atom": bond.GetEndAtomIdx(),
            "bond_type": str(bond.GetBondType()),
            "bond_order": bond.GetBondTypeAsDouble(),
            "aromatic": bond.GetIsAromatic(),
            "conjugated": bond.GetIsConjugated(),
            "in_ring": bond.IsInRing(),
        })

    return pd.DataFrame(bonds)

In [ ]:
bond_df = bond_features(mol)

bond_df

### Adjacency Matrix

In [ ]:
def molecular_adjacency_matrix(mol):
    n_atoms = mol.GetNumAtoms()

    adjacency = np.zeros(
        (n_atoms, n_atoms),
        dtype=np.float32
    )

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        adjacency[i, j] = 1
        adjacency[j, i] = 1

    return adjacency

In [ ]:
adj = molecular_adjacency_matrix(mol)

print(adj.shape)
print(adj)

### Weighted Adjacency Matrix

In [ ]:
def weighted_adjacency_matrix(mol):
    n_atoms = mol.GetNumAtoms()

    adjacency = np.zeros(
        (n_atoms, n_atoms),
        dtype=np.float32
    )

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        order = bond.GetBondTypeAsDouble()

        adjacency[i, j] = order
        adjacency[j, i] = order

    return adjacency

### Morgan Fingerprints

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [ ]:
def morgan_fingerprint(mol):
    return morgan_generator.GetFingerprint(mol)

In [ ]:
data["morgan_fp"] = data["mol"].apply(
    morgan_fingerprint
)

### Convert Morgan Fingerprints to NumPy

In [ ]:
def fingerprint_to_numpy(fp):
    arr = np.zeros(
        (fp.GetNumBits(),),
        dtype=np.uint8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    return arr

In [ ]:
valid_data["morgan_array"] = valid_data["morgan_fp"].apply(
    fingerprint_to_numpy
)

In [ ]:
X_morgan = np.vstack(
    valid_data["morgan_array"].values
)

print(X_morgan.shape)

### Morgan Fingerprint Information

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

additional_output = rdFingerprintGenerator.AdditionalOutput()

additional_output.CollectBitInfoMap()

mol = valid_data.iloc[0]["mol"]

fp = morgan_generator.GetFingerprint(
    mol,
    additionalOutput=additional_output
)

bit_info = additional_output.GetBitInfoMap()

print(bit_info)

### RDKit Fingerprint

In [ ]:
rdkit_generator = rdFingerprintGenerator.GetRDKitFPGenerator(
    fpSize=2048
)

valid_data["rdkit_fp"] = valid_data["mol"].apply(
    rdkit_generator.GetFingerprint
)

### Atom Pair Fingerprint

In [ ]:
atom_pair_generator = rdFingerprintGenerator.GetAtomPairGenerator(
    fpSize=2048
)

valid_data["atom_pair_fp"] = valid_data["mol"].apply(
    atom_pair_generator.GetFingerprint
)

### Topological Torsion Fingerprint

In [ ]:
torsion_generator = rdFingerprintGenerator.GetTopologicalTorsionGenerator(
    fpSize=2048
)

valid_data["torsion_fp"] = valid_data["mol"].apply(
    torsion_generator.GetFingerprint
)

### MACCS Fingerprint

In [ ]:
valid_data["maccs_fp"] = valid_data["mol"].apply(
    MACCSkeys.GenMACCSKeys
)

### Fingerprint Summary

In [ ]:
print("Morgan:", len(valid_data.iloc[0]["morgan_fp"]))
print("RDKit:", len(valid_data.iloc[0]["rdkit_fp"]))
print("Atom Pair:", len(valid_data.iloc[0]["atom_pair_fp"]))
print("Topological Torsion:", len(valid_data.iloc[0]["torsion_fp"]))
print("MACCS:", len(valid_data.iloc[0]["maccs_fp"]))

### Molecular Similarity

In [ ]:
mol_a = valid_data.iloc[0]["mol"]
mol_b = valid_data.iloc[1]["mol"]

fp_a = morgan_generator.GetFingerprint(mol_a)
fp_b = morgan_generator.GetFingerprint(mol_b)

similarity = DataStructs.TanimotoSimilarity(
    fp_a,
    fp_b
)

print("Tanimoto similarity:", similarity)

### Similarity Search

In [ ]:
query_mol = valid_data.iloc[0]["mol"]

query_fp = morgan_generator.GetFingerprint(
    query_mol
)

similarities = []

for idx, row in valid_data.iterrows():

    fp = morgan_generator.GetFingerprint(
        row["mol"]
    )

    score = DataStructs.TanimotoSimilarity(
        query_fp,
        fp
    )

    similarities.append({
        "index": idx,
        "Compound ID": row["Compound ID"],
        "smiles": row["smiles"],
        "similarity": score
    })

In [ ]:
similarity_df = pd.DataFrame(
    similarities
)

In [ ]:
similarity_df = similarity_df.sort_values(
    "similarity",
    ascending=False
)

In [ ]:
similarity_df.head(10)

### Similarity Matrix

In [ ]:
mols = valid_data["mol"].tolist()

fps = [
    morgan_generator.GetFingerprint(mol)
    for mol in mols
]

In [ ]:
n = len(fps)

similarity_matrix = np.zeros(
    (n, n),
    dtype=np.float32
)

In [ ]:
for i in range(n):
    similarity_matrix[i, i] = 1.0

    for j in range(i + 1, n):

        score = DataStructs.TanimotoSimilarity(
            fps[i],
            fps[j]
        )

        similarity_matrix[i, j] = score
        similarity_matrix[j, i] = score

In [ ]:
similarity_matrix.shape

### Drug-Likeness — Lipinski Rule of Five

In [ ]:
def lipinski_analysis(mol):

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    violations = {
        "MW": mw > 500,
        "LogP": logp > 5,
        "HBD": hbd > 5,
        "HBA": hba > 10,
    }

    violation_count = sum(
        violations.values()
    )

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "Ro5_Violations": violation_count,
        "Ro5_Pass": violation_count <= 1,
    }

In [ ]:
ro5_results = valid_data["mol"].apply(
    lipinski_analysis
)

ro5_df = pd.DataFrame(
    ro5_results.tolist(),
    index=valid_data.index
)

valid_data = pd.concat(
    [valid_data, ro5_df],
    axis=1
)

### QED Drug-Likeness

In [ ]:
valid_data["QED"] = valid_data["mol"].apply(
    QED.qed
)

In [ ]:
valid_data[
    [
        "Compound ID",
        "QED",
        "MW",
        "LogP",
        "HBD",
        "HBA",
        "Ro5_Violations"
    ]
].head()

### One Unified Molecular Analysis Function

In [ ]:
def analyze_molecule(mol):

    if mol is None:
        return None

    properties = calculate_molecular_properties(mol)

    ro5 = lipinski_analysis(mol)

    descriptors = Descriptors.CalcMolDescriptors(
        mol,
        missingVal=np.nan
    )

    return {
        "properties": properties,
        "ro5": ro5,
        "descriptors": descriptors,
        "QED": QED.qed(mol),
        "num_atoms": mol.GetNumAtoms(),
        "num_bonds": mol.GetNumBonds(),
        "num_rings": rdMolDescriptors.CalcNumRings(mol),
        "formal_charge": Chem.GetFormalCharge(mol),
        "canonical_smiles": Chem.MolToSmiles(mol),
    }

In [ ]:
result = analyze_molecule(
    valid_data.iloc[0]["mol"]
)

result["properties"]

### Complete Molecular Profile

In [ ]:
def molecular_profile(mol):

    if mol is None:
        raise ValueError("Invalid molecule")

    profile = {
        "Canonical SMILES": Chem.MolToSmiles(mol),

        # Structure
        "Atoms": mol.GetNumAtoms(),
        "Bonds": mol.GetNumBonds(),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "Formal Charge": Chem.GetFormalCharge(mol),

        # Properties
        "Molecular Weight": Descriptors.MolWt(mol),
        "Exact Molecular Weight": Descriptors.ExactMolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "Rotatable Bonds": Lipinski.NumRotatableBonds(mol),
        "Heavy Atoms": Lipinski.HeavyAtomCount(mol),
        "Fraction Csp3": rdMolDescriptors.CalcFractionCSP3(mol),
        "MolMR": Crippen.MolMR(mol),

        # Drug-likeness
        "QED": QED.qed(mol),
    }

    return profile

In [ ]:
profile = molecular_profile(
    valid_data.iloc[0]["mol"]
)

pd.Series(profile)

### 04. Dataset Statistics

* Number of molecules
* Missing values
* Duplicate molecules
* Unique SMILES
* Property ranges

In [35]:
print("Number of Molecules:", data["smiles"].nunique())

Number of Molecules: 1128


In [36]:
# Number of rows / records
print("Number of Rows:", len(data))

Number of Rows: 1128


In [37]:
# Number of unique molecules based on SMILES
print("Unique SMILES:", data["smiles"].nunique())

Unique SMILES: 1128


In [38]:
# Duplicate SMILES
print("Duplicate SMILES:", data["smiles"].duplicated().sum())

Duplicate SMILES: 0


In [42]:
# Missing values
print("\nMissing Values:")
print("*" * 55)
print(data.isnull().sum())


Missing Values:
*******************************************************
Compound ID                                        0
ESOL predicted log solubility in mols per litre    0
Minimum Degree                                     0
Molecular Weight                                   0
Number of H-Bond Donors                            0
Number of Rings                                    0
Number of Rotatable Bonds                          0
Polar Surface Area                                 0
measured log solubility in mols per litre          0
smiles                                             0
dtype: int64


In [45]:
# Property ranges
print("\nProperty Ranges:")

print("1. Molecular Weight:",
      data["Molecular Weight"].min(),
      "to",
      data["Molecular Weight"].max())

print("2. H-Bond Donors:",
      data["Number of H-Bond Donors"].min(),
      "to",
      data["Number of H-Bond Donors"].max())

print("3. Rings:",
      data["Number of Rings"].min(),
      "to",
      data["Number of Rings"].max())

print("4. Rotatable Bonds:",
      data["Number of Rotatable Bonds"].min(),
      "to",
      data["Number of Rotatable Bonds"].max())

print("5. Polar Surface Area:",
      data["Polar Surface Area"].min(),
      "to",
      data["Polar Surface Area"].max())

print("6. Measured logS:",
      data["measured log solubility in mols per litre"].min(),
      "to",
      data["measured log solubility in mols per litre"].max())

print("7. ESOL Predicted logS:",
      data["ESOL predicted log solubility in mols per litre"].min(),
      "to",
      data["ESOL predicted log solubility in mols per litre"].max())


print("8. Minimum Degree:",
      data["Minimum Degree"].min(),
      "to",
      data["Minimum Degree"].max())




Property Ranges:
1. Molecular Weight: 16.043 to 780.9490000000001
2. H-Bond Donors: 0 to 11
3. Rings: 0 to 8
4. Rotatable Bonds: 0 to 23
5. Polar Surface Area: 0.0 to 268.67999999999995
6. Measured logS: -11.6 to 1.58
7. ESOL Predicted logS: -9.702 to 1.091
8. Minimum Degree: 0 to 2


### 05. Data Cleaning

* Missing SMILES
* Invalid SMILES
* Duplicate molecules
* Invalid molecular structures

In [46]:
# missing smiles
print("\nMissing SMILES:", data["smiles"].isnull().sum())


Missing SMILES: 0


In [48]:
(data["smiles"].str.strip() == "").sum()

np.int64(0)

In [ ]:
# Convert Mol → SMILES

In [49]:
# Convert SMILES → RDKit Mol
from rdkit import Chem

data["mol"] = data["smiles"].apply(Chem.MolFromSmiles)

In [50]:
print(data["mol"].head())

0    <rdkit.Chem.rdchem.Mol object at 0x000001752F6...
1    <rdkit.Chem.rdchem.Mol object at 0x000001752F6...
2    <rdkit.Chem.rdchem.Mol object at 0x000001752F6...
3    <rdkit.Chem.rdchem.Mol object at 0x000001752F6...
4    <rdkit.Chem.rdchem.Mol object at 0x000001752F6...
Name: mol, dtype: object


In [66]:
data[["Compound ID", "smiles", "mol"]].head()

,Compound ID,smiles,mol
0,Amigdalin,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...
1,Fenfuram,Cc1occc1C(=O)Nc2ccccc2,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...
2,citral,CC(C)=CCCC(C)=CC(=O),<rdkit.Chem.rdchem.Mol object at 0x000001752F6...
3,Picene,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...
4,Thiophene,c1ccsc1,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...


In [51]:
# Invalid SMILES
invalid_mols = data["mol"].isnull().sum()

print("Invalid SMILES:", invalid_mols)

Invalid SMILES: 0


In [52]:
# duplicate molecules based on RDKit Mol
duplicate_mols = data["mol"].duplicated().sum()
print("Duplicate Molecules:", duplicate_mols)

Duplicate Molecules: 0


In [53]:
data[data["smiles"].duplicated(keep=False)].sort_values("smiles")

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles,mol


In [54]:
# canonical SMILES
data["canonical_smiles"] = data["mol"].apply(
    Chem.MolToSmiles
)

In [55]:
print(
    "Duplicate molecular structures:",
    data["canonical_smiles"].duplicated().sum()
)

Duplicate molecular structures: 11


In [58]:
# Remove Duplicate Molecular Structures
data = data.drop_duplicates(subset="canonical_smiles").copy()

In [60]:
data.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles,mol,canonical_smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...,N#CC(OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O)c...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...,Cc1occc1C(=O)Nc1ccccc1
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O),<rdkit.Chem.rdchem.Mol object at 0x000001752F6...,CC(C)=CCCC(C)=CC=O
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...,c1ccc2c(c1)ccc1c2ccc2c3ccccc3ccc21
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1,<rdkit.Chem.rdchem.Mol object at 0x000001752F6...,c1ccsc1


In [61]:
# print("Duplicate molecular structures after removal:", data)

### SMILES & Molecular Representation

In [ ]:
# Convert SMILES → Mol
data["mol"] = data["smiles"].apply(Chem.MolFromSmiles)

In [ ]:
print(data["mol"].head())

In [ ]:
# Convert Mol → SMILES
data["smiles"] = data["mol"].apply(Chem.MolToSmiles)

In [ ]:
print(data["smiles"].head())

In [75]:
# Check whether molecule is valid
data["is_valid"] = data["mol"].notnull()

In [76]:
# Validation summary
print("Total molecules:", len(data))
print("Valid molecules:", data["is_valid"].sum())
print("Invalid molecules:", (~data["is_valid"]).sum())

Total molecules: 1117
Valid molecules: 1117
Invalid molecules: 0


In [62]:
data['smiles'].head(10)

0    OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1                               Cc1occc1C(=O)Nc2ccccc2
2                                 CC(C)=CCCC(C)=CC(=O)
3                   c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4                                              c1ccsc1
5                                       c2ccc1scnc1c2 
6                   Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl
7                     CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O
8       ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
9     COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C 
Name: smiles, dtype: str

In [67]:
for name, smiles in zip(data['Compound ID'].head(10), data['smiles'].head(10)):
    print(f"{name:15} → {smiles}")

Amigdalin       → OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)C(O)C3O 
Fenfuram        → Cc1occc1C(=O)Nc2ccccc2
citral          → CC(C)=CCCC(C)=CC(=O)
Picene          → c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
Thiophene       → c1ccsc1
benzothiazole   → c2ccc1scnc1c2 
2,2,4,6,6'-PCB  → Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl
Estradiol       → CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O
Dieldrin        → ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
Rotenone        → COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C 


In [ ]:
# Molecular Representation Table

# total atoms
data["num_atoms"] = data["mol"].apply(lambda mol: mol.GetNumAtoms())

# total bonds
data["num_bonds"] = data["mol"].apply(lambda mol: mol.GetNumBonds())

# display the molecular representation table
data[["Compound ID", "smiles", "num_atoms", "num_bonds"]].head(10)

,Compound ID,smiles,num_atoms,num_bonds
0,Amigdalin,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,32,34
1,Fenfuram,Cc1occc1C(=O)Nc2ccccc2,15,16
2,citral,CC(C)=CCCC(C)=CC(=O),11,10
3,Picene,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,22,26
4,Thiophene,c1ccsc1,5,5
5,benzothiazole,c2ccc1scnc1c2,9,10
6,"2,2,4,6,6'-PCB",Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl,17,18
7,Estradiol,CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O,20,23
8,Dieldrin,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl,19,23
9,Rotenone,COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C,29,33


### 06. Understand SMILES

* Atoms
* Bonds
* Branches
* Ring notation
* Charges
* Stereochemistry
* Aromatic atoms

In [73]:
from rdkit import Chem

# Number of atoms
data["atoms_number"] = data["mol"].apply(
    lambda mol: mol.GetNumAtoms()
)

# Number of bonds
data["bonds_number"] = data["mol"].apply(
    lambda mol: mol.GetNumBonds()
)

# Number of rings
data["rings_number"] = data["mol"].apply(
    lambda mol: mol.GetRingInfo().NumRings()
)

# Formal charge
data["formal_charge"] = data["mol"].apply(
    Chem.GetFormalCharge
)

# Stereochemistry / chiral centers
data["chiral_centers"] = data["mol"].apply(
    lambda mol: Chem.FindMolChiralCenters(
        mol,
        includeUnassigned=True
    )
)

# Aromatic atom indices
data["aromatic_atoms"] = data["mol"].apply(
    lambda mol: [
        atom.GetIdx()
        for atom in mol.GetAtoms()
        if atom.GetIsAromatic()
    ]
)

In [74]:
data[
    [
        "Compound ID",
        "smiles",
        "atoms_number",
        "bonds_number",
        "rings_number",
        "formal_charge",
        "chiral_centers",
        "aromatic_atoms"
    ]
].head(10)

,Compound ID,smiles,atoms_number,bonds_number,rings_number,formal_charge,chiral_centers,aromatic_atoms
0,Amigdalin,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,32,34,3,0,"[(2, ?), (4, ?), (7, ?), (9, ?), (11, ?), (20,...","[14, 15, 16, 17, 18, 19]"
1,Fenfuram,Cc1occc1C(=O)Nc2ccccc2,15,16,2,0,[],"[1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14]"
2,citral,CC(C)=CCCC(C)=CC(=O),11,10,0,0,[],[]
3,Picene,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,22,26,5,0,[],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
4,Thiophene,c1ccsc1,5,5,1,0,[],"[0, 1, 2, 3, 4]"
5,benzothiazole,c2ccc1scnc1c2,9,10,2,0,[],"[0, 1, 2, 3, 4, 5, 6, 7, 8]"
6,"2,2,4,6,6'-PCB",Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl,17,18,2,0,[],"[1, 2, 3, 5, 6, 8, 9, 10, 12, 13, 14, 15]"
7,Estradiol,CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O,20,23,4,0,"[(1, ?), (4, ?), (5, ?), (15, ?), (18, ?)]","[8, 9, 10, 12, 13, 14]"
8,Dieldrin,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl,19,23,5,0,"[(4, ?), (6, ?), (7, ?), (9, ?), (10, ?), (12,...",[]
9,Rotenone,COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C,29,33,5,0,"[(7, ?), (12, ?), (20, ?)]","[2, 3, 4, 9, 10, 14, 15, 16, 17, 21, 22, 23]"


## Molecular Structure Exploration

> Now investigate what actually exists inside the molecule.

### 11. Explore Atoms
* Atomic number
* Element
* Symbol
* Degree
* Total degree
* Hybridization
* Formal charge
* Aromaticity
* Number of hydrogens
* Ring membership

In [ ]:
atom_records = []

for idx, row in valid_data.iterrows():

    mol = row["mol"]

    for atom in mol.GetAtoms():

        atom_records.append({
            "compound_id": row["Compound ID"],
            "smiles": row["smiles"],
            "atom_index": atom.GetIdx(),
            "atomic_number": atom.GetAtomicNum(),
            "element": atom.GetSymbol(),
            "symbol": atom.GetSymbol(),
            "degree": atom.GetDegree(),
            "total_degree": atom.GetTotalDegree(),
            "hybridization": str(atom.GetHybridization()),
            "formal_charge": atom.GetFormalCharge(),
            "aromatic": atom.GetIsAromatic(),
            "total_hydrogens": atom.GetTotalNumHs(),
            "ring_member": atom.IsInRing(),
        })

atom_dataset = pd.DataFrame(atom_records)

In [ ]:
atom_dataset.head(20)

In [ ]:
# Explore One Molecule

def inspect_atoms(mol):

    records = []

    for atom in mol.GetAtoms():

        records.append({
            "Index": atom.GetIdx(),
            "Element": atom.GetSymbol(),
            "Atomic Number": atom.GetAtomicNum(),
            "Degree": atom.GetDegree(),
            "Total Degree": atom.GetTotalDegree(),
            "Hybridization": str(atom.GetHybridization()),
            "Formal Charge": atom.GetFormalCharge(),
            "Aromatic": atom.GetIsAromatic(),
            "Hydrogens": atom.GetTotalNumHs(),
            "In Ring": atom.IsInRing(),
        })

    return pd.DataFrame(records)

In [ ]:
mol = Chem.MolFromSmiles("CCOc1ccccc1")

inspect_atoms(mol)

### Visualize Atom Indices

In [ ]:
from rdkit.Chem import Draw

mol = Chem.MolFromSmiles("CCOc1ccccc1")

Draw.MolToImage(
    mol,
    size=(500, 500),
    kekulize=True,
    legend="Atom indices"
)

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D

drawer = rdMolDraw2D.MolDraw2DCairo(600, 500)

drawer.drawOptions().addAtomIndices = True

drawer.DrawMolecule(mol)
drawer.FinishDrawing()

img = drawer.GetDrawingText()

with open("atom_indices.png", "wb") as f:
    f.write(img)

### 12. Explore Bonds
* Bond type
* Single
* Double
* Triple
* Aromatic
* Bond order
* Ring bond
* Conjugation

In [ ]:
bond_records = []

for idx, row in valid_data.iterrows():

    if mol is None:
        raise ValueError("Invalid molecule")

    mol = row["mol"]

    for bond in mol.GetBonds():

        bond_records.append({
            "compound_id": row["Compound ID"],
            "smiles": row["smiles"],

            "bond_index": bond.GetIdx(),

            "begin_atom": bond.GetBeginAtomIdx(),
            "end_atom": bond.GetEndAtomIdx(),

            "begin_element": bond.GetBeginAtom().GetSymbol(),
            "end_element": bond.GetEndAtom().GetSymbol(),

            "bond_type": str(bond.GetBondType()),
            "bond_order": bond.GetBondTypeAsDouble(),

            "aromatic": bond.GetIsAromatic(),
            "ring_bond": bond.IsInRing(),
            "conjugated": bond.GetIsConjugated(),
        })

bond_dataset = pd.DataFrame(bond_records)

In [ ]:
bond_dataset.head(20)

In [ ]:
mol = Chem.MolFromSmiles(
    "CC(=O)Oc1ccccc1C(=O)O"
)

bond_df = explore_bonds(mol)

bond_df

### 14. Explore Elements

> Calculate: and their frequencies.

```text
C
N
O
S
P
F
Cl
Br
I
...

### 15. Formal Charge
* Neutral molecules
* Cations
* Anions
* Charged atoms

### 16. Aromaticity
* Aromatic atoms
* Aromatic bonds
* Aromatic systems

### 19. Customize Visualization
* Atom labels
* Bond display
* Image size
* Highlighting atoms
* Highlighting bonds
* Highlighting functional groups

### 20. Export Molecular Images

```text
PNG
SVG
```

### 33. Basic Descriptors

```text
MW
LogP
TPSA
HBD
HBA
Rotatable Bonds
Heavy Atom Count
Ring Count
Aromatic Ring Count
Fraction Csp3
Formal Charge
MolMR
```

In [ ]:
from rdkit import Chem
from rdkit.Chem import (
    Descriptors,
    Crippen,
    Lipinski,
    rdMolDescriptors
)


def calculate_basic_descriptors(mol):

    if mol is None:
        return None

    return {
        "MW": Descriptors.MolWt(mol),

        "LogP": Crippen.MolLogP(mol),

        "TPSA": rdMolDescriptors.CalcTPSA(mol),

        "HBD": Lipinski.NumHDonors(mol),

        "HBA": Lipinski.NumHAcceptors(mol),

        "Rotatable_Bonds": (
            Lipinski.NumRotatableBonds(mol)
        ),

        "Heavy_Atom_Count": (
            Lipinski.HeavyAtomCount(mol)
        ),

        "Ring_Count": (
            rdMolDescriptors.CalcNumRings(mol)
        ),

        "Aromatic_Ring_Count": (
            rdMolDescriptors.CalcNumAromaticRings(mol)
        ),

        "Fraction_Csp3": (
            rdMolDescriptors.CalcFractionCSP3(mol)
        ),

        "Formal_Charge": (
            Chem.GetFormalCharge(mol)
        ),

        "MolMR": Crippen.MolMR(mol)
    }

In [ ]:
descriptor_results = valid_data["mol"].apply(
    calculate_basic_descriptors
)

descriptor_df = pd.DataFrame(
    descriptor_results.tolist(),
    index=valid_data.index
)

In [ ]:
descriptor_df.head()

In [ ]:
# Descriptor Summary
basic_descriptor_columns = [
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "Heavy_Atom_Count",
    "Ring_Count",
    "Aromatic_Ring_Count",
    "Fraction_Csp3",
    "Formal_Charge",
    "MolMR"
]

descriptor_summary = (
    valid_data[basic_descriptor_columns]
    .describe()
    .T
)

descriptor_summary

### 34. EState Descriptors
> EState = Electrotopological State) descriptors are molecular descriptors that combine:

1. Atom-level electronic information
2. Topological information — how atoms are connected in the molecular graph

```Text
EState
│
├── Atom-level
│   ├── EState value
│   ├── EState atom type
│   ├── Element
│   ├── Degree
│   ├── Hybridization
│   ├── Charge
│   ├── Aromaticity
│   ├── Hydrogens
│   └── Ring membership
│
└── Molecule-level
    ├── EState Sum
    ├── EState Mean
    ├── EState Min
    ├── EState Max
    ├── EState Std
    └── EState Range
```

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import EState

In [ ]:
# ------------------------------------------------------------
# 5. Display atom-level EState for one example molecule
# ------------------------------------------------------------

example_idx = valid_data.index[0]

example_mol = valid_data.loc[
    example_idx,
    "mol"
]

example_estate = EState.EStateIndices(
    example_mol
)

example_types = EState.TypeAtoms(
    example_mol
)


atom_estate_df = pd.DataFrame({
    "Atom_Index": [
        atom.GetIdx()
        for atom in example_mol.GetAtoms()
    ],

    "Element": [
        atom.GetSymbol()
        for atom in example_mol.GetAtoms()
    ],

    "Atomic_Number": [
        atom.GetAtomicNum()
        for atom in example_mol.GetAtoms()
    ],

    "Degree": [
        atom.GetDegree()
        for atom in example_mol.GetAtoms()
    ],

    "Hybridization": [
        str(atom.GetHybridization())
        for atom in example_mol.GetAtoms()
    ],

    "Formal_Charge": [
        atom.GetFormalCharge()
        for atom in example_mol.GetAtoms()
    ],

    "Aromatic": [
        atom.GetIsAromatic()
        for atom in example_mol.GetAtoms()
    ],

    "Ring_Member": [
        atom.IsInRing()
        for atom in example_mol.GetAtoms()
    ],

    "Hydrogens": [
        atom.GetTotalNumHs()
        for atom in example_mol.GetAtoms()
    ],

    "EState_Type": example_types,

    "EState": example_estate
})

In [ ]:

# ------------------------------------------------------------
# 6. Display example atom-level EState analysis
# ------------------------------------------------------------

print("=" * 70)
print("ATOM-LEVEL EState ANALYSIS")
print("=" * 70)

print(
    "Compound:",
    valid_data.loc[example_idx, "Compound ID"]
)

print(
    "SMILES:",
    valid_data.loc[example_idx, "smiles"]
)

display(atom_estate_df)

In [ ]:

# ------------------------------------------------------------
# 1. EState descriptor calculation for one molecule
# ------------------------------------------------------------

def calculate_estate_descriptors(mol):
    """
    Calculate atom-level EState values and
    aggregate them into molecule-level descriptors.
    """

    if mol is None:
        return None

    # Atom-level EState indices
    estate_values = np.asarray(
        EState.EStateIndices(mol),
        dtype=float
    )

    # EState atom types
    estate_types = EState.TypeAtoms(mol)

    # Aggregate molecular-level EState descriptors
    result = {
        "EState_Sum": np.sum(estate_values),
        "EState_Mean": np.mean(estate_values),
        "EState_Min": np.min(estate_values),
        "EState_Max": np.max(estate_values),
        "EState_Std": np.std(estate_values),
        "EState_Range": np.ptp(estate_values),

        # Keep the original atom-level representation
        "EState_Values": estate_values.tolist(),
        "EState_Atom_Types": estate_types
    }

    return result



In [ ]:
# ------------------------------------------------------------
# 3. Convert molecular-level EState descriptors to DataFrame
# ------------------------------------------------------------

estate_df = pd.DataFrame(
    estate_results.tolist(),
    index=valid_data.index
)


In [ ]:
# ------------------------------------------------------------
# 4. Add molecular-level EState descriptors to dataset
# ------------------------------------------------------------

valid_data = pd.concat(
    [
        valid_data,
        estate_df[
            [
                "EState_Sum",
                "EState_Mean",
                "EState_Min",
                "EState_Max",
                "EState_Std",
                "EState_Range"
            ]
        ]
    ],
    axis=1
)


In [ ]:


# ------------------------------------------------------------
# 7. Display molecular-level EState descriptors
# ------------------------------------------------------------

print("=" * 70)
print("MOLECULAR-LEVEL EState DESCRIPTORS")
print("=" * 70)

molecular_estate_columns = [
    "Compound ID",
    "smiles",
    "EState_Sum",
    "EState_Mean",
    "EState_Min",
    "EState_Max",
    "EState_Std",
    "EState_Range"
]

display(
    valid_data[molecular_estate_columns].head(10)
)

In [ ]:
# Statistical summary

print("=" * 70)
print("EState STATISTICAL SUMMARY")
print("=" * 70)

estate_summary = (
    valid_data[
        [
            "EState_Sum",
            "EState_Mean",
            "EState_Min",
            "EState_Max",
            "EState_Std",
            "EState_Range"
        ]
    ]
    .describe()
    .T
)

display(estate_summary)

In [ ]:
# visualize EState Distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    valid_data["EState_Mean"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("Mean EState")
plt.ylabel("Number of Molecules")
plt.title("Mean EState Distribution")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    valid_data["EState_Max"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("Maximum EState")
plt.ylabel("Number of Molecules")
plt.title("Maximum EState Distribution")

plt.tight_layout()
plt.show()

### 35. Chi Descriptors

```text
Chi0
Chi1
Chi2
...

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import GraphDescriptors

In [ ]:
# 1. Calculate Chi descriptors for one molecule
# ------------------------------------------------------------

def calculate_chi_descriptors(mol):

    if mol is None:
        return None

    return {
        # Standard Chi connectivity indices
        "Chi0": GraphDescriptors.Chi0(mol),
        "Chi1": GraphDescriptors.Chi1(mol),
        "Chi2": GraphDescriptors.Chi2(mol),

        # Normalized / valence variants
        "Chi0n": GraphDescriptors.Chi0n(mol),
        "Chi1n": GraphDescriptors.Chi1n(mol),
        "Chi2n": GraphDescriptors.Chi2n(mol),

        "Chi0v": GraphDescriptors.Chi0v(mol),
        "Chi1v": GraphDescriptors.Chi1v(mol),
        "Chi2v": GraphDescriptors.Chi2v(mol),
    }


In [ ]:
# 3. Calculate Chi descriptors
# ------------------------------------------------------------

chi_results = valid_data["mol"].apply(
    calculate_chi_descriptors
)

In [ ]:
# 4. Convert to DataFrame
# ------------------------------------------------------------

chi_df = pd.DataFrame(
    chi_results.tolist(),
    index=valid_data.index
)


In [ ]:
# 5. Add Chi descriptors to dataset
# ------------------------------------------------------------

valid_data = pd.concat(
    [
        valid_data,
        chi_df
    ],
    axis=1
)

In [ ]:
# 6. Display Chi descriptors
# ------------------------------------------------------------

chi_columns = [
    "Chi0",
    "Chi1",
    "Chi2",
    "Chi0n",
    "Chi1n",
    "Chi2n",
    "Chi0v",
    "Chi1v",
    "Chi2v"
]

print("=" * 70)
print("CHI DESCRIPTORS")
print("=" * 70)

display(
    valid_data[
        ["Compound ID", "smiles"] + chi_columns
    ].head(10)
)


In [ ]:
# 7. Statistical summary
# ------------------------------------------------------------

print("=" * 70)
print("CHI DESCRIPTOR STATISTICAL SUMMARY")
print("=" * 70)

chi_summary = (
    valid_data[chi_columns]
    .describe()
    .T
)

display(chi_summary)

In [ ]:
# 9. Correlation between Chi descriptors
# ------------------------------------------------------------

print("=" * 70)
print("CHI DESCRIPTOR CORRELATION")
print("=" * 70)

chi_correlation = (
    valid_data[chi_columns]
    .corr()
    .round(3)
)

display(chi_correlation)


In [ ]:
# 10. Create a clean Chi feature matrix
# ------------------------------------------------------------

chi_feature_matrix = valid_data[
    ["Compound ID"] + chi_columns
].copy()

print("=" * 70)
print("CHI FEATURE MATRIX")
print("=" * 70)

print(
    "Shape:",
    chi_feature_matrix.shape
)

display(
    chi_feature_matrix.head(10)
)

### 36. Kappa Descriptors

> Kappa descriptors are molecular shape and connectivity indices developed from molecular graph theory. They are commonly used as classical molecular descriptors in QSAR/QSPR.

The three primary Kappa descriptors are:

- Kappa1
- Kappa2
- Kappa3


In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import GraphDescriptors

In [ ]:
# 1. Calculate Kappa descriptors for one molecule
# ------------------------------------------------------------

def calculate_kappa_descriptors(mol):

    if mol is None:
        return None

    return {
        "Kappa1": GraphDescriptors.Kappa1(mol),
        "Kappa2": GraphDescriptors.Kappa2(mol),
        "Kappa3": GraphDescriptors.Kappa3(mol),
    }

In [ ]:
# Calculate Kappa descriptors

kappa_results = valid_data["mol"].apply(
    calculate_kappa_descriptors
)

In [ ]:
# Convert results to DataFrame

kappa_df = pd.DataFrame(
    kappa_results.tolist(),
    index=valid_data.index
)

In [ ]:
# Add Kappa descriptors to dataset

valid_data = pd.concat(
    [
        valid_data,
        kappa_df
    ],
    axis=1
)

In [ ]:
# Display Kappa descriptors

kappa_columns = [
    "Kappa1",
    "Kappa2",
    "Kappa3"
]

print("=" * 70)
print("KAPPA DESCRIPTORS")
print("=" * 70)

display(
    valid_data[
        ["Compound ID", "smiles"] + kappa_columns
    ].head(10)
)

In [ ]:
# Statistical summary

print("=" * 70)
print("KAPPA DESCRIPTOR STATISTICAL SUMMARY")
print("=" * 70)

kappa_summary = (
    valid_data[kappa_columns]
    .describe()
    .T
)

display(kappa_summary)

In [ ]:
# Kappa descriptor correlation
print("=" * 70)
print("KAPPA DESCRIPTOR CORRELATION")
print("=" * 70)

kappa_correlation = (
    valid_data[kappa_columns]
    .corr()
    .round(3)
)

display(kappa_correlation)

In [ ]:
# Create clean Kappa feature matrix

kappa_feature_matrix = valid_data[
    ["Compound ID"] + kappa_columns
].copy()

print("=" * 70)
print("KAPPA FEATURE MATRIX")
print("=" * 70)

print(
    "Shape:",
    kappa_feature_matrix.shape
)

display(
    kappa_feature_matrix.head(10)
)

In [ ]:
# Inspect one molecule

example_idx = valid_data.index[0]

print("=" * 70)
print("EXAMPLE MOLECULE")
print("=" * 70)

print(
    "Compound:",
    valid_data.loc[example_idx, "Compound ID"]
)

print(
    "SMILES:",
    valid_data.loc[example_idx, "smiles"]
)

for descriptor in kappa_columns:

    print(
        f"{descriptor:8s}: "
        f"{valid_data.loc[example_idx, descriptor]:.6f}"
    )

### 37. BalabanJ Descriptor
> BalabanJ is a topological molecular descriptor that characterizes molecular connectivity using the molecular graph and distance/connectivity information. It is useful in QSAR/QSPR as a compact descriptor of molecular topology.

In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import GraphDescriptors

In [ ]:
# 1. Calculate BalabanJ for one molecule
# ------------------------------------------------------------

def calculate_balaban_j(mol):

    if mol is None:
        return np.nan

    return GraphDescriptors.BalabanJ(mol)

In [ ]:
# 3. Calculate BalabanJ
# ------------------------------------------------------------

valid_data["BalabanJ"] = valid_data["mol"].apply(
    calculate_balaban_j
)

In [ ]:
# 4. Display BalabanJ values
# ------------------------------------------------------------

print("=" * 70)
print("BALABAN J DESCRIPTOR")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BalabanJ"
        ]
    ].head(10)
)

In [ ]:
# 5. Statistical summary
# ------------------------------------------------------------

print("=" * 70)
print("BALABANJ STATISTICAL SUMMARY")
print("=" * 70)

balaban_summary = (
    valid_data["BalabanJ"]
    .describe()
    .to_frame()
    .T
)

display(balaban_summary)

In [ ]:
# 6. Check missing values
# ------------------------------------------------------------

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)

print(
    "Missing BalabanJ:",
    valid_data["BalabanJ"].isna().sum()
)

In [ ]:
# Basic distribution analysis
print("=" * 70)
print("BALABANJ RANGE")
print("=" * 70)

print(
    "Minimum:",
    valid_data["BalabanJ"].min()
)

print(
    "Maximum:",
    valid_data["BalabanJ"].max()
)

print(
    "Mean:",
    valid_data["BalabanJ"].mean()
)

print(
    "Median:",
    valid_data["BalabanJ"].median()
)

In [ ]:
# Distribution plot

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    valid_data["BalabanJ"].dropna(),
    bins=30,
    edgecolor="black"
)

plt.xlabel("BalabanJ")
plt.ylabel("Number of Molecules")
plt.title("BalabanJ Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect molecules with lowest BalabanJ
print("=" * 70)
print("LOWEST BALABANJ VALUES")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BalabanJ"
        ]
    ]
    .nsmallest(10, "BalabanJ")
)

In [ ]:
# Inspect molecules with highest BalabanJ
print("=" * 70)
print("HIGHEST BALABANJ VALUES")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BalabanJ"
        ]
    ]
    .nlargest(10, "BalabanJ")
)

In [ ]:
# Create BalabanJ feature matrix
balaban_feature_matrix = valid_data[
    [
        "Compound ID",
        "BalabanJ"
    ]
].copy()


print("=" * 70)
print("BALABANJ FEATURE MATRIX")
print("=" * 70)

print(
    "Shape:",
    balaban_feature_matrix.shape
)

display(
    balaban_feature_matrix.head(10)
)

In [ ]:
# Correlation with selected molecular descriptors
correlation_columns = [
    "BalabanJ"
]

# Add columns only if they already exist
for column in [
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Ring_Count",
    "Rotatable_Bonds",
    "Fraction_Csp3"
]:
    if column in valid_data.columns:
        correlation_columns.append(column)


print("=" * 70)
print("BALABANJ CORRELATION ANALYSIS")
print("=" * 70)

display(
    valid_data[correlation_columns]
    .corr()
    .round(3)
)

### 38. BertzCT

> Explore molecular complexity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import GraphDescriptors

In [ ]:
# Calculate BertzCT for one molecule
def calculate_bertzct(mol):

    if mol is None:
        return np.nan

    return GraphDescriptors.BertzCT(mol)

In [ ]:
# Calculate BertzCT
valid_data["BertzCT"] = valid_data["mol"].apply(
    calculate_bertzct
)

In [ ]:
# Display BertzCT

print("=" * 70)
print("BERTZCT DESCRIPTOR")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BertzCT"
        ]
    ].head(10)
)

In [ ]:
# Statistical summary

print("=" * 70)
print("BERTZCT STATISTICAL SUMMARY")
print("=" * 70)

bertz_summary = (
    valid_data["BertzCT"]
    .describe()
    .to_frame()
    .T
)

display(bertz_summary)

In [ ]:
# Missing values

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)

print(
    "Missing BertzCT:",
    valid_data["BertzCT"].isna().sum()
)

In [ ]:
# Basic statistics
print("=" * 70)
print("BERTZCT RANGE")
print("=" * 70)

print("Minimum :", valid_data["BertzCT"].min())
print("Maximum :", valid_data["BertzCT"].max())
print("Mean    :", valid_data["BertzCT"].mean())
print("Median  :", valid_data["BertzCT"].median())
print("Std     :", valid_data["BertzCT"].std())

In [ ]:
# BertzCT distribution

plt.figure(figsize=(8, 5))

plt.hist(
    valid_data["BertzCT"].dropna(),
    bins=30,
    edgecolor="black"
)

plt.xlabel("BertzCT")
plt.ylabel("Number of Molecules")
plt.title("BertzCT Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Lowest BertzCT molecules

print("=" * 70)
print("LOWEST BERTZCT VALUES")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BertzCT"
        ]
    ]
    .nsmallest(10, "BertzCT")
)


In [ ]:
# Highest BertzCT molecules

print("=" * 70)
print("HIGHEST BERTZCT VALUES")
print("=" * 70)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "BertzCT"
        ]
    ]
    .nlargest(10, "BertzCT")
)

In [ ]:
# Correlation with other descriptors

correlation_columns = ["BertzCT"]

for column in [
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "Ring_Count",
    "Rotatable_Bonds",
    "Fraction_Csp3",
    "BalabanJ",
    "Kappa1",
    "Kappa2",
    "Kappa3"
]:

    if column in valid_data.columns:
        correlation_columns.append(column)


print("=" * 70)
print("BERTZCT CORRELATION ANALYSIS")
print("=" * 70)

display(
    valid_data[correlation_columns]
    .corr()
    .round(3)
)

In [ ]:
# Create BertzCT feature matrix

bertz_feature_matrix = valid_data[
    [
        "Compound ID",
        "BertzCT"
    ]
].copy()


print("=" * 70)
print("BERTZCT FEATURE MATRIX")
print("=" * 70)

print(
    "Shape:",
    bertz_feature_matrix.shape
)

display(
    bertz_feature_matrix.head(10)
)

In [ ]:
# Example molecule

example_idx = valid_data.index[0]

print("=" * 70)
print("EXAMPLE MOLECULE")
print("=" * 70)

print(
    "Compound:",
    valid_data.loc[example_idx, "Compound ID"]
)

print(
    "SMILES:",
    valid_data.loc[example_idx, "smiles"]
)

print(
    "BertzCT:",
    valid_data.loc[example_idx, "BertzCT"]
)

## Molecular Fingerprints
* Morgan Fingerprints

* ECFP Concept
        ```text
        ECFP4
        ECFP6
        ```


* Fingerprint Bit Representation

* Fingerprint Visualization

    > Visualize which molecular fragments activate fingerprint bits.

* Other Fingerprints
    * MACCS
    * RDKit fingerprints
    * Atom-pair fingerprints
    * Topological torsion fingerprints

* Molecular Similarity

In [ ]:
from rdkit import Chem
from rdkit.Chem import (
    rdFingerprintGenerator,
    MACCSkeys
)


morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

rdkit_generator = rdFingerprintGenerator.GetRDKitFPGenerator(
    fpSize=2048
)

atom_pair_generator = (
    rdFingerprintGenerator.GetAtomPairGenerator(
        fpSize=2048
    )
)

torsion_generator = (
    rdFingerprintGenerator.GetTopologicalTorsionGenerator(
        fpSize=2048
    )
)


def generate_fingerprints(mol):

    return {
        "Morgan": morgan_generator.GetFingerprint(mol),

        "MACCS": MACCSkeys.GenMACCSKeys(mol),

        "RDKit": rdkit_generator.GetFingerprint(mol),

        "AtomPair": atom_pair_generator.GetFingerprint(mol),

        "TopologicalTorsion":
            torsion_generator.GetFingerprint(mol),
    }

In [ ]:
fps = generate_fingerprints(mol)

for name, fp in fps.items():
    print(
        f"{name:20} → {len(fp)} bits"
    )

### Store Fingerprints in Dataset

In [ ]:
valid_data["morgan_fp"] = valid_data["mol"].apply(
    morgan_generator.GetFingerprint
)

valid_data["maccs_fp"] = valid_data["mol"].apply(
    MACCSkeys.GenMACCSKeys
)

valid_data["rdkit_fp"] = valid_data["mol"].apply(
    rdkit_generator.GetFingerprint
)

valid_data["atom_pair_fp"] = valid_data["mol"].apply(
    atom_pair_generator.GetFingerprint
)

valid_data["torsion_fp"] = valid_data["mol"].apply(
    torsion_generator.GetFingerprint
)

In [ ]:
# Molecular Similarity

mol_a = valid_data.iloc[0]["mol"]
mol_b = valid_data.iloc[1]["mol"]

fp_a = morgan_generator.GetFingerprint(mol_a)
fp_b = morgan_generator.GetFingerprint(mol_b)

similarity = DataStructs.TanimotoSimilarity(
    fp_a,
    fp_b
)

print("Tanimoto similarity:", similarity)


In [ ]:
# Similarity Search

query_mol = valid_data.iloc[0]["mol"]

query_fp = morgan_generator.GetFingerprint(
    query_mol
)


In [ ]:
similarities = []

for idx, row in valid_data.iterrows():

    fp = morgan_generator.GetFingerprint(
        row["mol"]
    )

    score = DataStructs.TanimotoSimilarity(
        query_fp,
        fp
    )

    similarities.append({
        "index": idx,
        "Compound ID": row["Compound ID"],
        "SMILES": row["smiles"],
        "Tanimoto": score
    })

In [ ]:
similarity_df = (
    pd.DataFrame(similarities)
    .sort_values(
        "Tanimoto",
        ascending=False
    )
)

In [ ]:
similarity_df.head(10)

### 50. Descriptor Distribution

* MW distribution
* LogP distribution
* TPSA distribution
* HBD distribution
* HBA distribution
* Ring count distribution

In [ ]:
# Molecular Weight Distribution
plt.figure(figsize=(8, 5))

plt.hist(
    summary_df["MW"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("Molecular Weight (Da)")
plt.ylabel("Number of Molecules")
plt.title("Molecular Weight Distribution")

plt.show()

In [ ]:
summary_df["MW"].describe()

In [ ]:
# LogP Distribution

plt.figure(figsize=(8, 5))

plt.hist(
    summary_df["LogP"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("LogP")
plt.ylabel("Number of Molecules")
plt.title("LogP Distribution")

plt.show()

In [ ]:
summary_df["LogP"].describe()

In [ ]:
summary_df.nlargest(
    10,
    "LogP"
)[
    ["Compound ID", "SMILES", "LogP"]
]

In [ ]:
summary_df.nsmallest(
    10,
    "LogP"
)[
    ["Compound ID", "SMILES", "LogP"]
]

In [ ]:
# TPSA Distribution
plt.figure(figsize=(8, 5))

plt.hist(
    summary_df["TPSA"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("TPSA (Å²)")
plt.ylabel("Number of Molecules")
plt.title("TPSA Distribution")

plt.show()

In [ ]:
summary_df["TPSA"].describe()

In [ ]:
summary_df.nlargest(
    10,
    "TPSA"
)[
    ["Compound ID", "SMILES", "TPSA"]
]

In [ ]:
# HBD Distribution
hbd_counts = summary_df["HBD"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

plt.bar(
    hbd_counts.index,
    hbd_counts.values
)

plt.xlabel("Hydrogen Bond Donors (HBD)")
plt.ylabel("Number of Molecules")
plt.title("HBD Distribution")

plt.xticks(hbd_counts.index)

plt.show()


In [ ]:
hbd_counts

In [ ]:
# HBA Distribution
hba_counts = summary_df["HBA"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

plt.bar(
    hba_counts.index,
    hba_counts.values
)

plt.xlabel("Hydrogen Bond Acceptors (HBA)")
plt.ylabel("Number of Molecules")
plt.title("HBA Distribution")

plt.xticks(hba_counts.index)

plt.show()

In [ ]:
summary_df["HBA"].describe()

In [ ]:
# Ring Count Distribution

ring_counts = summary_df["Rings"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

plt.bar(
    ring_counts.index,
    ring_counts.values
)

plt.xlabel("Number of Rings")
plt.ylabel("Number of Molecules")
plt.title("Ring Count Distribution")

plt.xticks(ring_counts.index)

plt.show()


In [ ]:
ring_counts

In [ ]:
continuous_descriptors = [
    ("MW", "Molecular Weight (Da)"),
    ("LogP", "LogP"),
    ("TPSA", "TPSA (Å²)")
]

discrete_descriptors = [
    ("HBD", "Hydrogen Bond Donors"),
    ("HBA", "Hydrogen Bond Acceptors"),
    ("Rings", "Number of Rings")
]

### 51. Property Correlations

```text
MW ↔ LogP
MW ↔ TPSA
LogP ↔ ESOL
TPSA ↔ ESOL
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Select required columns
# ------------------------------------------------------------

# ESOL predicted logS from the original dataset
esol_column = "ESOL_LogS"

# If your summary_df contains the ESOL column but valid_data
# does not, copy it across.
if esol_column not in valid_data.columns:

    if esol_column in summary_df.columns:
        valid_data[esol_column] = summary_df.loc[
            valid_data.index,
            esol_column
        ]

    else:
        valid_data[esol_column] = valid_data[
            "ESOL predicted log solubility in mols per litre"
        ]


correlation_columns = [
    "MW",
    "LogP",
    "TPSA",
    "ESOL_LogS"
]


# ------------------------------------------------------------
# 2. Create clean correlation DataFrame
# ------------------------------------------------------------

correlation_df = valid_data[
    correlation_columns
].copy()

correlation_df = correlation_df.dropna()


print("=" * 70)
print("PROPERTY CORRELATION DATA")
print("=" * 70)

print(
    "Number of molecules:",
    len(correlation_df)
)

display(
    correlation_df.head(10)
)


# ------------------------------------------------------------
# 3. Pearson correlation
# ------------------------------------------------------------

pearson_corr = correlation_df.corr(
    method="pearson"
)


print("=" * 70)
print("PEARSON CORRELATION")
print("=" * 70)

display(
    pearson_corr.round(3)
)


# ------------------------------------------------------------
# 4. Spearman correlation
# ------------------------------------------------------------

spearman_corr = correlation_df.corr(
    method="spearman"
)


print("=" * 70)
print("SPEARMAN CORRELATION")
print("=" * 70)

display(
    spearman_corr.round(3)
)

In [ ]:
 Full correlation matrix visualization
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.imshow(
    pearson_corr,
    interpolation="nearest",
    aspect="auto"
)

plt.colorbar(
    label="Pearson Correlation"
)

plt.xticks(
    range(len(correlation_columns)),
    correlation_columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(correlation_columns)),
    correlation_columns
)

plt.title(
    "Molecular Property Correlation Matrix"
)

# Add correlation values
for i in range(len(correlation_columns)):
    for j in range(len(correlation_columns)):

        value = pearson_corr.iloc[i, j]

        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


In [ ]:
# 6. Scatter plot function
# ------------------------------------------------------------

def plot_correlation(
    df,
    x,
    y,
    xlabel=None,
    ylabel=None,
    title=None
):

    x_values = df[x]
    y_values = df[y]

    pearson = x_values.corr(
        y_values,
        method="pearson"
    )

    spearman = x_values.corr(
        y_values,
        method="spearman"
    )

    plt.figure(figsize=(8, 5))

    plt.scatter(
        x_values,
        y_values,
        alpha=0.6
    )

    plt.xlabel(
        xlabel if xlabel else x
    )

    plt.ylabel(
        ylabel if ylabel else y
    )

    plt.title(
        title
        if title
        else f"{x} vs {y}"
    )

    plt.text(
        0.05,
        0.95,
        f"Pearson r = {pearson:.3f}\n"
        f"Spearman ρ = {spearman:.3f}",
        transform=plt.gca().transAxes,
        verticalalignment="top"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# 12. Identify strongest pairwise correlation
# ------------------------------------------------------------

corr_matrix = pearson_corr.copy()

# Remove diagonal
np.fill_diagonal(
    corr_matrix.values,
    np.nan
)

strongest_pair = (
    corr_matrix
    .abs()
    .stack()
    .sort_values(
        ascending=False
    )
)

print("=" * 70)
print("STRONGEST PAIRWISE CORRELATIONS")
print("=" * 70)

display(
    strongest_pair.head(6)
)

### 52. Outlier Detection
> For your Molecular Structure & Property Explorer, this section should identify molecules that are unusual compared with the rest of the ESOL dataset.

* Extremely high MW
* Extremely high LogP
* Extremely high TPSA
* Unusual charge
* Unusual ring systems

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

In [ ]:
# Make sure required descriptors exist

if "MW" not in valid_data.columns:
    valid_data["MW"] = valid_data["mol"].apply(
        lambda mol: rdMolDescriptors.CalcExactMolWt(mol)
    )

if "LogP" not in valid_data.columns:
    from rdkit.Chem import Crippen

    valid_data["LogP"] = valid_data["mol"].apply(
        Crippen.MolLogP
    )

if "TPSA" not in valid_data.columns:
    valid_data["TPSA"] = valid_data["mol"].apply(
        rdMolDescriptors.CalcTPSA
    )

if "Formal_Charge" not in valid_data.columns:
    valid_data["Formal_Charge"] = valid_data["mol"].apply(
        Chem.GetFormalCharge
    )

if "Ring_Count" not in valid_data.columns:
    valid_data["Ring_Count"] = valid_data["mol"].apply(
        rdMolDescriptors.CalcNumRings
    )

if "Aromatic_Ring_Count" not in valid_data.columns:
    valid_data["Aromatic_Ring_Count"] = valid_data["mol"].apply(
        rdMolDescriptors.CalcNumAromaticRings
    )

In [ ]:
# IQR-based outlier detection

def iqr_bounds(series):

    series = series.dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return lower, upper

In [ ]:
# MW outliers

mw_lower, mw_upper = iqr_bounds(
    valid_data["MW"]
)

valid_data["MW_Outlier"] = (
    (valid_data["MW"] < mw_lower) |
    (valid_data["MW"] > mw_upper)
)

In [ ]:
# LogP outliers

logp_lower, logp_upper = iqr_bounds(
    valid_data["LogP"]
)

valid_data["LogP_Outlier"] = (
    (valid_data["LogP"] < logp_lower) |
    (valid_data["LogP"] > logp_upper)
)

In [ ]:
# TPSA outliers

tpsa_lower, tpsa_upper = iqr_bounds(
    valid_data["TPSA"]
)

valid_data["TPSA_Outlier"] = (
    (valid_data["TPSA"] < tpsa_lower) |
    (valid_data["TPSA"] > tpsa_upper)
)

In [ ]:
# Unusual charge

valid_data["Unusual_Charge"] = (
    valid_data["Formal_Charge"] != 0
)

In [ ]:
# Unusual ring systems

ring_lower, ring_upper = iqr_bounds(
    valid_data["Ring_Count"]
)

valid_data["Unusual_Ring_System"] = (
    (valid_data["Ring_Count"] < ring_lower) |
    (valid_data["Ring_Count"] > ring_upper)
)

In [ ]:
# Overall outlier flag

outlier_flags = [
    "MW_Outlier",
    "LogP_Outlier",
    "TPSA_Outlier",
    "Unusual_Charge",
    "Unusual_Ring_System"
]

valid_data["Any_Outlier"] = (
    valid_data[outlier_flags]
    .any(axis=1)
)

In [ ]:
# Outlier score

valid_data["Outlier_Count"] = (
    valid_data[outlier_flags]
    .sum(axis=1)
)

In [ ]:
# Display threshold information

print("=" * 75)
print("OUTLIER THRESHOLDS")
print("=" * 75)

print(f"MW    : < {mw_lower:.3f} or > {mw_upper:.3f}")
print(f"LogP  : < {logp_lower:.3f} or > {logp_upper:.3f}")
print(f"TPSA  : < {tpsa_lower:.3f} or > {tpsa_upper:.3f}")
print(f"Rings : < {ring_lower:.3f} or > {ring_upper:.3f}")


In [ ]:
# Summary of detected outliers

print("\n" + "=" * 75)
print("OUTLIER SUMMARY")
print("=" * 75)

outlier_summary = pd.DataFrame({
    "Category": [
        "MW",
        "LogP",
        "TPSA",
        "Unusual Charge",
        "Unusual Ring System"
    ],

    "Number of Outliers": [
        valid_data["MW_Outlier"].sum(),
        valid_data["LogP_Outlier"].sum(),
        valid_data["TPSA_Outlier"].sum(),
        valid_data["Unusual_Charge"].sum(),
        valid_data["Unusual_Ring_System"].sum()
    ]
})

outlier_summary["Percentage"] = (
    outlier_summary["Number of Outliers"]
    / len(valid_data)
    * 100
)

display(
    outlier_summary.round(2)
)

In [ ]:
# Extremely high MW molecules

print("=" * 75)
print("EXTREMELY HIGH MOLECULAR WEIGHT")
print("=" * 75)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "MW",
            "LogP",
            "TPSA"
        ]
    ]
    .sort_values(
        "MW",
        ascending=False
    )
    .head(10)
)

In [ ]:
# Extremely high LogP molecules

print("=" * 75)
print("EXTREMELY HIGH LOGP")
print("=" * 75)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "MW",
            "LogP",
            "TPSA"
        ]
    ]
    .sort_values(
        "LogP",
        ascending=False
    )
    .head(10)
)

In [ ]:
# Extremely high TPSA molecules

print("=" * 75)
print("EXTREMELY HIGH TPSA")
print("=" * 75)

display(
    valid_data[
        [
            "Compound ID",
            "smiles",
            "MW",
            "LogP",
            "TPSA"
        ]
    ]
    .sort_values(
        "TPSA",
        ascending=False
    )
    .head(10)
)

In [ ]:

# Unusual charge

print("=" * 75)
print("UNUSUAL CHARGE")
print("=" * 75)

display(
    valid_data[
        valid_data["Unusual_Charge"]
    ][
        [
            "Compound ID",
            "smiles",
            "Formal_Charge",
            "MW",
            "LogP",
            "TPSA"
        ]
    ]
    .sort_values(
        "Formal_Charge"
    )
    .head(20)
)


In [ ]:

# Unusual ring systems

print("=" * 75)
print("UNUSUAL RING SYSTEMS")
print("=" * 75)

display(
    valid_data[
        valid_data["Unusual_Ring_System"]
    ][
        [
            "Compound ID",
            "smiles",
            "Ring_Count",
            "Aromatic_Ring_Count",
            "MW",
            "LogP"
        ]
    ]
    .sort_values(
        "Ring_Count",
        ascending=False
    )
    .head(20)
)

In [ ]:

# Molecules with multiple outlier flags

print("=" * 75)
print("MOLECULES WITH MULTIPLE OUTLIER FLAGS")
print("=" * 75)

multi_outliers = (
    valid_data[
        valid_data["Outlier_Count"] >= 2
    ][
        [
            "Compound ID",
            "smiles",
            "MW",
            "LogP",
            "TPSA",
            "Formal_Charge",
            "Ring_Count",
            "Outlier_Count"
        ] + outlier_flags
    ]
    .sort_values(
        "Outlier_Count",
        ascending=False
    )
)

display(
    multi_outliers.head(20)
)

In [ ]:

# Outlier distribution visualization

plt.figure(figsize=(8, 5))

plt.bar(
    outlier_summary["Category"],
    outlier_summary["Number of Outliers"]
)

plt.xlabel("Outlier Category")
plt.ylabel("Number of Molecules")
plt.title("Molecular Outlier Summary")

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:

# MW vs LogP — highlight outliers

plt.figure(figsize=(8, 5))

normal = ~valid_data["Any_Outlier"]
outlier = valid_data["Any_Outlier"]

plt.scatter(
    valid_data.loc[normal, "MW"],
    valid_data.loc[normal, "LogP"],
    alpha=0.5,
    label="Non-outlier"
)

plt.scatter(
    valid_data.loc[outlier, "MW"],
    valid_data.loc[outlier, "LogP"],
    alpha=0.8,
    label="Outlier"
)

plt.xlabel("Molecular Weight (Da)")
plt.ylabel("LogP")
plt.title("MW vs LogP — Outlier Detection")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final outlier dataset

outlier_dataset = valid_data[
    valid_data["Any_Outlier"]
].copy()

print("=" * 75)
print("FINAL OUTLIER DATASET")
print("=" * 75)

print(
    "Total valid molecules:",
    len(valid_data)
)

print(
    "Molecules with ≥1 outlier flag:",
    len(outlier_dataset)
)

print(
    "Percentage:",
    f"{len(outlier_dataset) / len(valid_data) * 100:.2f}%"
)

display(
    outlier_dataset[
        [
            "Compound ID",
            "smiles",
            "MW",
            "LogP",
            "TPSA",
            "Formal_Charge",
            "Ring_Count",
            "Aromatic_Ring_Count",
            "Outlier_Count"
        ] + outlier_flags
    ].head(20)
)

## Finally, Build the Molecule Explorer Application